# S02 — Descriptive Analysis

Phase 1 capstone. Five zones (BE, FI, SG, US-MIDA-PJM, US-NY-NYIS), hourly data 2021-01 through 2025-12. Read-only over `data/raw/`; produces figures bound for the report's Data section and the per-region heterogeneity atlas (Contribution 3).

## 1. Setup

In [ ]:
from __future__ import annotations

from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import carbon_forecast
from carbon_forecast.data.storage import read_parquet
from carbon_forecast.data.zones import ZONES
from carbon_forecast.plotting.config import (
    ENERGY_PALETTE,
    PLOT_H,
    PLOT_W,
    REGIONAL_PALETTE,
    apply_defaults,
    style_fig,
)

apply_defaults()

PROJECT_ROOT = Path(carbon_forecast.__file__).resolve().parents[2]
DATA_ROOT = PROJECT_ROOT / "data"
ZONE_KEYS = [z.em_key for z in ZONES]

print("Zones    :", ZONE_KEYS)
print("Data root:", DATA_ROOT)

## 2. Long-format master frames

One loader per (zone, endpoint) family. Each concatenates every monthly Parquet on disk into one long DataFrame indexed by UTC time. Cached in memory via `lru_cache` so subsequent cells reuse the same frames.

In [ ]:
def _load_endpoint(zone: str, endpoint: str) -> pd.DataFrame:
    root = DATA_ROOT / "raw/em" / zone / endpoint
    if not root.exists():
        return pd.DataFrame()
    frames = [read_parquet(f) for f in sorted(root.glob("*.parquet"))]
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


@lru_cache(maxsize=None)
def load_ci_long() -> pd.DataFrame:
    frames = []
    for zone in ZONE_KEYS:
        df = _load_endpoint(zone, "carbon-intensity/past-range")
        if df.empty:
            continue
        df = df.copy()
        df["zone"] = zone
        frames.append(df)
    return pd.concat(frames) if frames else pd.DataFrame()


@lru_cache(maxsize=None)
def load_pb(zone: str) -> pd.DataFrame:
    return _load_endpoint(zone, "power-breakdown/past-range")


@lru_cache(maxsize=None)
def load_flows(zone: str) -> pd.DataFrame:
    return _load_endpoint(zone, "electricity-flows/past-range")


@lru_cache(maxsize=None)
def load_weather(zone: str) -> pd.DataFrame:
    root = DATA_ROOT / "raw/weather" / zone
    if not root.exists():
        return pd.DataFrame()
    frames = [read_parquet(f) for f in sorted(root.glob("*.parquet"))]
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

In [ ]:
ci_long = load_ci_long()
print("CI long shape :", ci_long.shape)
print("Zones present :", sorted(ci_long["zone"].unique()))
print("Date range    :", ci_long.index.min(), "to", ci_long.index.max())
ci_long.head()

## 3. CI distributions per zone

Summary statistics (5th, 50th, 95th percentile plus mean / std) followed by a violin plot. Visual ranking should follow the rough fuel-mix story: FI lowest (nuclear + hydro), BE next (mixed), US zones in the middle, SG highest and tightest (gas-dominated, no renewables).

In [ ]:
ci_summary = (
    ci_long.groupby("zone")["carbon_intensity_gco2eq_kwh"]
    .describe(percentiles=[0.05, 0.5, 0.95])
    .round(1)
)
ci_summary

In [ ]:
fig = px.violin(
    ci_long,
    x="zone",
    y="carbon_intensity_gco2eq_kwh",
    color="zone",
    color_discrete_map=REGIONAL_PALETTE,
    category_orders={"zone": ZONE_KEYS},
    box=True,
    points=False,
    labels={"carbon_intensity_gco2eq_kwh": "Carbon intensity (gCO2eq/kWh)", "zone": "Zone"},
)
fig.update_layout(showlegend=False)
style_fig(fig, "Carbon intensity distribution by zone, 2021 to 2025")
fig.show()

## 4. Annual seasonality

Monthly-mean CI per zone, averaged across all five years. Reveals which zones swing across the calendar (winter heating, summer cooling, hydro snowmelt, solar season).

In [ ]:
monthly = (
    ci_long.assign(month=lambda d: d.index.month)
    .groupby(["zone", "month"])["carbon_intensity_gco2eq_kwh"]
    .mean()
    .reset_index()
)

fig = px.line(
    monthly,
    x="month",
    y="carbon_intensity_gco2eq_kwh",
    color="zone",
    color_discrete_map=REGIONAL_PALETTE,
    category_orders={"zone": ZONE_KEYS},
    markers=True,
    labels={"month": "Month", "carbon_intensity_gco2eq_kwh": "Mean CI (gCO2eq/kWh)", "zone": "Zone"},
)
fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(1, 13)),
    ticktext=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
)
style_fig(fig, "Mean carbon intensity by month, averaged over 2021 to 2025")
fig.show()

## 5. Weekly seasonality

Day-of-week by hour-of-day heatmap per zone. Reveals weekend vs weekday dispatch patterns. Strong demand-driven zones (US, BE) should show weekday peaks; weakly-cyclic zones (SG) should look flat.

In [ ]:
DAY_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def dow_hod(df: pd.DataFrame) -> pd.DataFrame:
    g = (
        df.assign(dow=df.index.dayofweek, hod=df.index.hour)
          .groupby(["dow", "hod"])["carbon_intensity_gco2eq_kwh"]
          .mean()
          .unstack("hod")
    )
    g.index = [DAY_LABELS[i] for i in g.index]
    return g

fig = make_subplots(
    rows=1, cols=5,
    subplot_titles=ZONE_KEYS,
    horizontal_spacing=0.04,
    shared_yaxes=True,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    sub = ci_long[ci_long["zone"] == zone].drop(columns="zone")
    heat = dow_hod(sub)
    fig.add_trace(
        go.Heatmap(
            z=heat.values,
            x=list(heat.columns),
            y=list(heat.index),
            colorscale="YlOrRd",
            showscale=(i == 5),
            colorbar=dict(title="gCO2eq/kWh") if i == 5 else None,
        ),
        row=1, col=i,
    )
    fig.update_xaxes(title_text="Hour (UTC)", row=1, col=i)
fig.update_yaxes(title_text="Day", row=1, col=1)
style_fig(fig, "Carbon intensity heatmap, day-of-week by hour-of-day, per zone")
fig.update_layout(width=int(PLOT_W * 1.6), height=PLOT_H)
fig.show()

## 6. Diurnal cycle

Mean hourly profile per zone with a shaded ±1 standard deviation band. Sharp evening peaks suggest demand-driven gas dispatch; flat profiles suggest baseload-dominated mixes.

In [ ]:
hourly = (
    ci_long.assign(hod=ci_long.index.hour)
    .groupby(["zone", "hod"])["carbon_intensity_gco2eq_kwh"]
    .agg(["mean", "std"])
    .reset_index()
)

fig = go.Figure()
for zone in ZONE_KEYS:
    z = hourly[hourly["zone"] == zone]
    color = REGIONAL_PALETTE[zone]
    band = hex_to_rgba(color, 0.15)
    fig.add_trace(go.Scatter(
        x=z["hod"], y=z["mean"] + z["std"],
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=z["hod"], y=z["mean"] - z["std"],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor=band,
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=z["hod"], y=z["mean"],
        mode="lines", name=zone,
        line=dict(color=color, width=2),
    ))
fig.update_xaxes(title_text="Hour of day (UTC)", tickmode="array", tickvals=list(range(0, 24, 2)))
fig.update_yaxes(title_text="Mean carbon intensity (gCO2eq/kWh)")
style_fig(fig, "Diurnal carbon intensity cycle, mean ± 1 std, per zone")
fig.show()

## 7. Production mix evolution

Annual mean production per source, stacked per zone, 2021 to 2025. Reveals 5-year transitions (e.g. wind growth, coal phase-out, nuclear changes).

In [ ]:
PROD_ORDER = [
    "nuclear", "biomass", "hydro", "wind", "solar",
    "hydro_discharge", "battery_discharge",
    "gas", "oil", "coal", "unknown",
]

fig = make_subplots(
    rows=1, cols=len(ZONE_KEYS),
    subplot_titles=ZONE_KEYS,
    horizontal_spacing=0.04,
    shared_yaxes=False,
)
seen_sources: set[str] = set()
for col_i, zone in enumerate(ZONE_KEYS, start=1):
    pb = load_pb(zone)
    if pb.empty:
        continue
    prod_cols = [c for c in pb.columns if c.startswith("prod_") and c.endswith("_mw")]
    src = pb[prod_cols].copy()
    src.columns = [c[len("prod_"):-len("_mw")] for c in prod_cols]
    annual = src.groupby(src.index.year).mean()
    annual = annual.reindex(columns=[c for c in PROD_ORDER if c in annual.columns])
    for source in annual.columns:
        show_legend = source not in seen_sources
        seen_sources.add(source)
        fig.add_trace(
            go.Bar(
                x=annual.index, y=annual[source],
                name=source,
                marker_color=ENERGY_PALETTE.get(source, "#888"),
                legendgroup=source,
                showlegend=show_legend,
            ),
            row=1, col=col_i,
        )
    fig.update_xaxes(title_text="Year", row=1, col=col_i, tickmode="linear", dtick=1)
fig.update_yaxes(title_text="Mean MW", row=1, col=1)
fig.update_layout(barmode="stack")
style_fig(fig, "Annual mean production mix per zone, 2021 to 2025")
fig.update_layout(width=int(PLOT_W * 1.6), height=PLOT_H)
fig.show()

## 8. Cross-border flows summary

Annual gross imports and exports per zone (TWh, averaged over years). Sets up the interconnection-regime ranking that Contribution 2 builds on: SG near-zero (control), FI and BE high, US zones moderate.

In [ ]:
rows = []
for zone in ZONE_KEYS:
    fl = load_flows(zone)
    if fl.empty:
        continue
    imp_cols = [c for c in fl.columns if c.startswith("import_") and c.endswith("_mw")]
    exp_cols = [c for c in fl.columns if c.startswith("export_") and c.endswith("_mw")]
    # MW summed across hourly index equals MWh (1-hour buckets).
    gross_imp_mwh = fl[imp_cols].fillna(0.0).sum().sum()
    gross_exp_mwh = fl[exp_cols].fillna(0.0).sum().sum()
    years_span = max((fl.index.max() - fl.index.min()).days / 365.25, 1e-6)
    rows.append({
        "zone": zone,
        "annual_gross_import_twh": gross_imp_mwh / 1e6 / years_span,
        "annual_gross_export_twh": gross_exp_mwh / 1e6 / years_span,
    })
flows_summary = pd.DataFrame(rows)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=flows_summary["zone"], y=flows_summary["annual_gross_import_twh"],
    name="Imports", marker_color="#3B7DD8",
))
fig.add_trace(go.Bar(
    x=flows_summary["zone"], y=flows_summary["annual_gross_export_twh"],
    name="Exports", marker_color="#D55E00",
))
fig.update_yaxes(title_text="Annual TWh")
fig.update_xaxes(title_text="Zone")
fig.update_layout(barmode="group")
style_fig(fig, "Annual gross cross-border flows per zone, TWh")
fig.show()

flows_summary.round(2)

## 9. Weather vs renewables — spot check

BE only: hourly wind speed against `prod_wind_mw`, and hourly shortwave radiation against `prod_solar_mw`. 10k-point random sample for plot speed. We expect a clear monotone (roughly cubic for wind) signal — if the scatter is shapeless, the weather feature isn't doing its job.

In [ ]:
be_pb = load_pb("BE")
be_wx = load_weather("BE")
joined = be_pb.join(be_wx, how="inner")
sample = joined.sample(n=min(10000, len(joined)), random_state=42).sort_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Wind speed vs wind production", "Solar irradiance vs solar production"),
    horizontal_spacing=0.12,
)
fig.add_trace(
    go.Scatter(
        x=sample["wind_speed_10m"], y=sample["prod_wind_mw"],
        mode="markers", marker=dict(size=3, opacity=0.25, color="#3B7DD8"),
        showlegend=False,
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=sample["shortwave_radiation"], y=sample["prod_solar_mw"],
        mode="markers", marker=dict(size=3, opacity=0.25, color="#F4D03F"),
        showlegend=False,
    ),
    row=1, col=2,
)
fig.update_xaxes(title_text="Wind speed (m/s)", row=1, col=1)
fig.update_yaxes(title_text="Wind production (MW)", row=1, col=1)
fig.update_xaxes(title_text="Shortwave radiation (W/m^2)", row=1, col=2)
fig.update_yaxes(title_text="Solar production (MW)", row=1, col=2)
style_fig(fig, "Belgium — weather drivers vs renewable production")
fig.update_layout(width=int(PLOT_W * 1.3), height=PLOT_H)
fig.show()

## 10. Findings

(Fill in after reviewing the figures.)

- **CI distributions:** 
- **Annual seasonality:** 
- **Weekly seasonality:** 
- **Diurnal cycle:** 
- **Production mix evolution:** 
- **Cross-border flows:** 
- **Weather correlations:** 

These findings feed the report's Data section and the per-region heterogeneity atlas (Contribution 3).